# Week 6 — Feature Engineering
**Internship:** IDX Exchange Data Science Program  
**Name:** Monika  
**Week:** 6  
**Dataset:** CRMLS Sold Properties, cleaned in Week 3, models from Week 5

**Goal:** Engineer additional property features and add a school district 
regional feature via spatial join, then compare model performance with and 
without the new features.

Two new features are added this week:
1. **BedBathRatio** — bedrooms divided by bathrooms
2. **SchoolDistrict** — which Unified School District each property falls 
   into, based on its coordinates

In [29]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

data_folder = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\data\california'
model_df = pd.read_csv(data_folder + '\\cleaned_full.csv', parse_dates=['CloseDate_parsed'])

BASE_FEATURE_COLS = [
    'LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres',
    'PropertyAge', 'DaysOnMarket', 'Latitude', 'Longitude',
    'PoolPrivateYN', 'ViewYN', 'WaterfrontYN', 'BasementYN', 'AssociationFee',
    'LivingArea_missing', 'BathroomsTotalInteger_missing',
    'YearBuilt_missing', 'LotSizeAcres_missing', 'DaysOnMarket_anomaly'
]
target_col = 'ClosePrice'
BEST_WINDOW = 3  # <-- set to whatever notebook 4 found best

print(f'Loaded {len(model_df):,} rows')

Loaded 411,419 rows


## 1. Engineer BedBathRatio
A simple ratio feature — properties with more bedrooms per bathroom may be 
priced differently than balanced or bathroom-heavy properties. Missing/undefined 
ratios (from 0 bathrooms) are filled with the median so no rows are lost.

In [30]:
model_df['BedBathRatio'] = model_df['BedroomsTotal'] / model_df['BathroomsTotalInteger'].replace(0, np.nan)
model_df['BedBathRatio'] = model_df['BedBathRatio'].fillna(model_df['BedBathRatio'].median())

## 2. Load School District Boundaries
Loading the CA School District Areas GeoJSON and inspecting its columns before 
doing anything else, since the exact column names in real open-data files don't 
always match documentation exactly.

In [31]:
district_path = r'C:\Users\monik\OneDrive - University of Illinois - Urbana\Desktop\IDX Exchange_DS\scripts\DistrictAreas2526_-284845464123469011.geojson'
districts = gpd.read_file(district_path)

print(districts.columns.tolist())
districts.head(2)

['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictName', 'DistrictType', 'GradeLow', 'GradeHigh', 'GradeLowCensus', 'GradeHighCensus', 'AssistStatus', 'UpdateNotes', 'EnrollTotal', 'EnrollCharter', 'EnrollNonCharter', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAreaSqMi', 'LocaleCode', 'LocaleDesc', 'geometry']


,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictName,DistrictType,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAreaSqMi,LocaleCode,LocaleDesc,geometry
0,1,2025-26,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1302,12.1,4259,39.5,11.248886,21,"21 - Suburban, Large","MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,2,2025-26,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,363,9.7,1247,33.3,1.789975,21,"21 - Suburban, Large","POLYGON ((-13612893.866 4565099.707, -13612896..."


## 3. Filter to Unified Districts and Align Coordinate Systems
Per instructions, only `DistrictType = "Unified"` districts are used. 

Important: the district file's coordinates are in a projected system (EPSG:3857), 
not standard lat/long degrees (EPSG:4326). Both datasets must share the same 
coordinate reference system (CRS) before a spatial join will work correctly, so 
properties are reprojected to match.

In [32]:
print(f'Districts CRS: {districts.crs}')

districts_unified = districts[districts['DistrictType'] == 'Unified'].copy()
print(f'Unified districts: {len(districts_unified):,} of {len(districts):,}')

properties_gdf = gpd.GeoDataFrame(
    model_df,
    geometry=gpd.points_from_xy(model_df['Longitude'], model_df['Latitude']),
    crs='EPSG:4326'  # our lat/long is standard WGS84 degrees
)

# reproject properties to match the district file's CRS (whatever it printed above)
properties_gdf = properties_gdf.to_crs(districts_unified.crs)
print(f'Properties reprojected to: {properties_gdf.crs}')

Districts CRS: EPSG:3857
Unified districts: 345 of 936
Properties reprojected to: EPSG:3857


## 4. Spatial Join: Match Each Property to Its District
Using `gpd.sjoin` with `predicate='within'` to find which Unified district 
polygon (if any) contains each property's coordinate point. Properties that 
fall outside all Unified boundaries (served by separate elementary/high-school 
districts instead) are labeled `'No_Unified_District'` rather than dropped.

In [33]:
joined = gpd.sjoin(
    properties_gdf,
    districts_unified[['DistrictName', 'geometry']],
    how='left',
    predicate='within'
)

n_matched = joined['DistrictName'].notna().sum()
print(f'Matched: {n_matched:,} / {len(joined):,} ({n_matched/len(joined)*100:.2f}%)')

model_df['SchoolDistrict'] = joined['DistrictName'].values
model_df['SchoolDistrict'] = model_df['SchoolDistrict'].fillna('No_Unified_District')

print(f'Rows before dedup check: {len(model_df):,}')
model_df = model_df[~model_df.index.duplicated(keep='first')]
print(f'Rows after dedup check: {len(model_df):,}')

Matched: 307,973 / 411,419 (74.86%)
Rows before dedup check: 411,419
Rows after dedup check: 411,419


## 5. One-Hot Encode SchoolDistrict and Build Train/Test Sets
SchoolDistrict is a categorical feature with 323 possible values, so it's 
one-hot encoded (converted into binary columns). One category is dropped as a 
reference to avoid the "dummy variable trap" — without this, Linear Regression 
becomes numerically unstable due to perfect multicollinearity between district 
columns.

In [34]:
train_df, test_df = get_train_test_split(model_df, test_month, BEST_WINDOW)

# One-hot encode SchoolDistrict -- fit categories on train only, then align test to match
train_dummies = pd.get_dummies(train_df['SchoolDistrict'], prefix='Dist', drop_first=True)
test_dummies = pd.get_dummies(test_df['SchoolDistrict'], prefix='Dist', drop_first=True)
train_dummies, test_dummies = train_dummies.align(test_dummies, join='left', axis=1, fill_value=0)

train_df = pd.concat([train_df.reset_index(drop=True), train_dummies.reset_index(drop=True)], axis=1)
test_df = pd.concat([test_df.reset_index(drop=True), test_dummies.reset_index(drop=True)], axis=1)

DISTRICT_COLS = train_dummies.columns.tolist()
print(f'Number of district dummy columns: {len(DISTRICT_COLS)}')

OLD_FEATURE_COLS = BASE_FEATURE_COLS
NEW_FEATURE_COLS = BASE_FEATURE_COLS + ['BedBathRatio'] + DISTRICT_COLS

Number of district dummy columns: 292


## 6. Retrain Models: Old Features vs. New Features
Comparing all three models (Linear Regression, Decision Tree, Random Forest) 
trained with and without the new features (BedBathRatio + SchoolDistrict), 
using the same window and test month as prior weeks for a fair comparison.

In [35]:
def evaluate_models(train_df, test_df, feature_cols, target_col):
    X_train, y_train = train_df[feature_cols], train_df[target_col]
    X_test, y_test = test_df[feature_cols], test_df[target_col]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    lr = LinearRegression().fit(X_train_scaled, y_train)
    dt = DecisionTreeRegressor(max_depth=10, random_state=42).fit(X_train, y_train)
    rf = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5,
                                random_state=42, n_jobs=-1).fit(X_train, y_train)

    return {
        'Linear Regression': r2_score(y_test, lr.predict(X_test_scaled)),
        'Decision Tree': r2_score(y_test, dt.predict(X_test)),
        'Random Forest': r2_score(y_test, rf.predict(X_test)),
    }

old_results = evaluate_models(train_df, test_df, OLD_FEATURE_COLS, target_col)
new_results = evaluate_models(train_df, test_df, NEW_FEATURE_COLS, target_col)

comparison_df = pd.DataFrame({
    'model': list(old_results.keys()),
    'R2_old_features': list(old_results.values()),
    'R2_new_features': [new_results[m] for m in old_results.keys()],
})
comparison_df['R2_improvement'] = comparison_df['R2_new_features'] - comparison_df['R2_old_features']
comparison_df

,model,R2_old_features,R2_new_features,R2_improvement
0,Linear Regression,0.481007,0.649220,0.168214
1,Decision Tree,0.784613,0.774022,-0.010591
2,Random Forest,0.873122,0.875719,0.002597


## Summary of Findings

- Engineered BedBathRatio as an additional property characteristic feature. 
  (PropertyAge was already engineered during Week 3 preprocessing and is 
  included in both the old and new feature sets below.)
- Spatially joined all properties to CA Unified School District boundaries via 
  geopandas sjoin (74.86% of properties matched to a Unified district; the 
  remainder fall under separate elementary/high-school district systems and 
  were labeled 'No_Unified_District').
- One-hot encoded SchoolDistrict (322 categories, dropping one reference 
  category to avoid multicollinearity in Linear Regression).
- Retrained all three models (Linear Regression, Decision Tree, Random Forest) 
  on the same 3-month training window used throughout Weeks 4-5, comparing old 
  vs. new feature sets on the June 2026 test month:

    | Model              | R2 (old) | R2 (new) | Improvement |
    |--------------------|----------|----------|-------------|
    | Linear Regression  | 0.481    | 0.649    | +0.168      |
    | Decision Tree       | 0.785    | 0.774    | -0.011      |
    | Random Forest       | 0.873    | 0.876    | +0.003      |

- Key insight: the school district feature meaningfully improved Linear 
  Regression, but had negligible-to-slightly-negative effect on tree-based 
  models. This makes sense given the model architectures -- Linear Regression 
  cannot capture nonlinear geographic patterns from raw Latitude/Longitude 
  alone, so explicit district labels provide new information it couldn't 
  otherwise learn. Decision Tree and Random Forest, however, can already split 
  on Latitude/Longitude to approximate geographic clusters, making the 322 
  additional district columns largely redundant signal -- and for Decision 
  Tree specifically, the added columns slightly hurt performance, likely by 
  giving the single tree more ways to overfit to noise in the training data.
- Random Forest remains the best-performing model overall (R2 = 0.876).